# FSL-Vision Data Pipeline — Fase 1 (definitiva)
**Trabajo Final Integrador — Inteligencia Computacional (IC415)**

Pipeline completo de datos para **Open-Set Recognition** sobre imágenes de ROV de **FathomNet**: extracción → AutoEDA → limpieza de corruptos → split estratificado sin *data leakage* → sanity check visual. Deja el dataset listo para entrenar.

> Ejecutar en orden **1 → 7**. Todo persiste en Drive; la extracción es idempotente (re-correr no vuelve a bajar lo ya descargado).

## Celda 1 — Preparación del Entorno

In [ ]:
# === CELDA 1: Preparación del Entorno ===

# 1) Montar Google Drive (acá persiste TODO: dataset, manifiesto y reporte).
from google.colab import drive
drive.mount('/content/drive')

# 2) Instalar dependencias.
#    - fathomnet: cliente oficial de FathomNet (MBARI)
#    - pyyaml: lectura del archivo de configuración
#    - ydata-profiling: motor de AutoEDA
#    - scikit-learn: split estratificado (Celda 6)
#    - Pillow: control de integridad y recortes (Celdas 5 y 7)
!pip install -q fathomnet pyyaml ydata-profiling scikit-learn Pillow

# NOTA COLAB: si la Celda 4 falla al importar ydata_profiling, reiniciá la
# sesión ("Entorno de ejecución -> Reiniciar sesión") y re-corré desde la 1.

# 3) Rutas centralizadas (evita inconsistencias entre celdas).
import os
PROJECT_DIR     = "/content/drive/MyDrive/IC415"          # <-- tu carpeta en Drive
DATA_RAW        = os.path.join(PROJECT_DIR, "data", "raw")
DATA_PROCESSED  = os.path.join(PROJECT_DIR, "data", "processed")
CONFIG_PATH     = os.path.join(PROJECT_DIR, "paraense_fauna.yaml")
MANIFIESTO      = os.path.join(DATA_RAW, "manifiesto_dataset.csv")
MANIFIESTO_LIMPIO = os.path.join(DATA_RAW, "manifiesto_limpio.csv")
REPORTE_EDA     = os.path.join(DATA_RAW, "reporte_eda.html")

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)
print("Proyecto:", PROJECT_DIR)
print("Crudo ->", DATA_RAW)
print("Procesado ->", DATA_PROCESSED)


## Celda 2 — Configuración (`paraense_fauna.yaml`)

La ruta del `%%writefile` debe coincidir con `PROJECT_DIR` de la Celda 1.

In [ ]:
%%writefile /content/drive/MyDrive/IC415/paraense_fauna.yaml
# =====================================================================
# paraense_fauna.yaml  -  Configuración del FSL-Vision Data Pipeline
# Fuente: FathomNet (imágenes de ROV de aguas profundas, MBARI et al.)
# Las 4 clases de "normalidad" (inliers) del Open-Set Recognition.
# =====================================================================

descarga:
  proveedor_taxonomia: "fathomnet"   # expande cada concepto a sus descendientes (server-side)
  solo_verificadas: true             # análogo a "research grade": solo anotaciones validadas
  imagenes_por_concepto: 400         # tope por clase
  pausa_segundos: 0.15               # cortesía entre descargas
  reintentos: 3                      # reintentos por imagen ante fallos de red

# Cada concepto genera una subcarpeta data/raw/<carpeta>/
conceptos:
  - carpeta: "asteroidea"
    concepto: "Asteroidea"     # estrellas de mar
  - carpeta: "scyphozoa"
    concepto: "Scyphozoa"      # medusas verdaderas
  - carpeta: "porifera"
    concepto: "Porifera"       # esponjas
  - carpeta: "octopoda"
    concepto: "Octopoda"       # pulpos


## Celda 3 — Extracción desde FathomNet + Manifiesto

Concepto + descendientes (server-side), solo anotaciones verificadas, idempotencia, reintentos con backoff y manifiesto de trazabilidad.

In [ ]:
# === CELDA 3: Extracción desde FathomNet + Manifiesto ===
import os, csv, time, requests, yaml
from fathomnet.api import images
from fathomnet.dto import GeoImageConstraints

HEADERS = {"User-Agent": "IC415-UNaM-FSL-Vision/1.0"}
# ROVs/plataformas conocidas; se detectan en la ruta de la URL (metadato extra útil).
PLATAFORMAS = ["Tiburon", "Ventana", "Doc Ricketts", "i2MAP", "MiniROV", "Hercules"]


def plataforma_desde_url(url):
    """Heurística: intenta inferir el vehículo/plataforma a partir de la URL."""
    if not url:
        return None
    u = url.replace(" ", "")
    for p in PLATAFORMAS:
        if p.replace(" ", "") in u:
            return p
    return None


def buscar_imagenes(concepto, cfg):
    """Busca en FathomNet imágenes del concepto y sus descendientes taxonómicos.
    Devuelve una lista de objetos imagen (AImageDTO)."""
    restriccion = GeoImageConstraints(
        concept=concepto,
        taxaProviderName=cfg["proveedor_taxonomia"],   # expande a descendientes
        includeVerified=cfg["solo_verificadas"],       # solo anotaciones validadas
        includeUnverified=not cfg["solo_verificadas"],
        limit=cfg["imagenes_por_concepto"],
    )
    return images.find(restriccion)


def descargar_imagen(url, ruta, reintentos, pausa):
    """Descarga robusta con reintentos y backoff. Devuelve True/False."""
    for intento in range(reintentos):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            with open(ruta, "wb") as f:
                f.write(r.content)
            time.sleep(pausa)
            return True
        except Exception as e:
            if intento == reintentos - 1:
                print(f"   ! descarga fallida ({e}) -> {url[:70]}")
                return False
            time.sleep(1.5 * (intento + 1))   # backoff creciente
    return False


def metadatos(img, concepto, carpeta, archivo):
    """Extrae del objeto imagen los metadatos útiles de FathomNet para el manifiesto."""
    return {
        "uuid": img.uuid,
        "concepto": concepto,
        "url_origen": img.url,
        "profundidad_m": img.depthMeters,
        "temperatura_c": img.temperatureCelsius,
        "imaging_type": img.imagingType,
        "plataforma": plataforma_desde_url(img.url),
        "institucion": (img.contributorsEmail or "").split("@")[-1] or None,
        "ancho": img.width,
        "alto": img.height,
        "lat": img.latitude,
        "lon": img.longitude,
        "archivo_local": os.path.join(carpeta, archivo),
    }


def extraer_clase(concepto, carpeta, cfg, vistos):
    """Descarga las imágenes de un concepto y devuelve las filas del manifiesto."""
    destino = os.path.join(DATA_RAW, carpeta)
    os.makedirs(destino, exist_ok=True)
    print(f"\n=== {carpeta} ({concepto}) ===")

    try:
        imgs = buscar_imagenes(concepto, cfg)
    except Exception as e:
        print(f"   ! búsqueda fallida para '{concepto}': {e}")
        return []
    print(f"   {len(imgs)} imágenes candidatas devueltas por FathomNet")

    filas, n = [], 0
    for img in imgs:
        # Saltar sin URL o ya usadas en otra clase (evita la misma imagen en 2 carpetas).
        if not img.url or img.uuid in vistos:
            continue
        archivo = f"{img.uuid}.jpg"
        ruta = os.path.join(destino, archivo)

        # IDEMPOTENCIA: si ya está en disco, no se vuelve a bajar (resume tras desconexión).
        if not os.path.exists(ruta):
            if not descargar_imagen(img.url, ruta, cfg["reintentos"], cfg["pausa_segundos"]):
                continue  # si falló la descarga, no registramos la fila

        vistos.add(img.uuid)
        filas.append(metadatos(img, concepto, carpeta, archivo))
        n += 1
        if n % 50 == 0:
            print(f"   ... {n} descargadas")
    print(f"   LISTO: {n} imágenes en {destino}")
    return filas


# ------------------------- Flujo principal -------------------------
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)
cfg = config["descarga"]

campos = ["uuid", "concepto", "url_origen", "profundidad_m", "temperatura_c",
          "imaging_type", "plataforma", "institucion", "ancho", "alto",
          "lat", "lon", "archivo_local"]

todas_las_filas = []
vistos = set()   # uuids ya descargados (dedupe global entre clases)
for c in config["conceptos"]:
    todas_las_filas += extraer_clase(c["concepto"], c["carpeta"], cfg, vistos)

with open(MANIFIESTO, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=campos)
    w.writeheader()
    w.writerows(todas_las_filas)

print(f"\n==> Manifiesto: {MANIFIESTO}  ({len(todas_las_filas)} filas)")


## Celda 4 — AutoEDA

Auditoría del dataset crudo: balance de clases, completitud de metadatos, duplicados.

In [ ]:
# === CELDA 4: AutoEDA ===
import pandas as pd
from ydata_profiling import ProfileReport   # puede emitir un warning de deprecación: es inofensivo

df = pd.read_csv(MANIFIESTO)
print("Dimensiones del manifiesto:", df.shape)
print("\nImágenes por concepto:\n", df["concepto"].value_counts())

# Completitud de metadatos: clave para este dataset, ya que FathomNet
# no siempre trae profundidad/temperatura/plataforma.
print("\nCompletitud por columna (%):")
print((df.notna().mean() * 100).round(1))

perfil = ProfileReport(
    df,
    title="AutoEDA — Dataset de Normalidad FathomNet (FSL-Vision)",
    explorative=True,
)
perfil.to_file(REPORTE_EDA)
print(f"\n==> Reporte EDA guardado en: {REPORTE_EDA}")

# Para verlo embebido en el notebook (opcional):
# perfil.to_notebook_iframe()


## Celda 5 — Control de Integridad

Abre cada imagen con PIL (`verify` + `load`); elimina del disco y del manifiesto las corruptas o truncadas. Imprime cuántas sobrevivieron.

In [ ]:
# === CELDA 5: Control de Integridad (limpieza de corruptos) ===
import os
import pandas as pd
from PIL import Image

def imagen_valida(ruta):
    """True si la imagen se abre y decodifica COMPLETA.
    - verify(): detecta cabeceras/estructura rotas.
    - load(): fuerza la decodificación real de los píxeles, lo que atrapa
      archivos TRUNCADOS (típico de descargas cortadas por desconexión en Colab).
    Se usan dos aperturas porque verify() deja el objeto inutilizable."""
    try:
        with Image.open(ruta) as im:
            im.verify()
        with Image.open(ruta) as im:
            im.load()
        return True
    except Exception:
        return False

df = pd.read_csv(MANIFIESTO)
n_inicial = len(df)

filas_ok, eliminadas = [], 0
for _, fila in df.iterrows():
    ruta = os.path.join(DATA_RAW, fila["archivo_local"])
    if os.path.exists(ruta) and imagen_valida(ruta):
        filas_ok.append(fila)
    else:
        # Corrupta o faltante: se borra del disco y se descarta del manifiesto.
        if os.path.exists(ruta):
            os.remove(ruta)
        eliminadas += 1

df_limpio = pd.DataFrame(filas_ok).reset_index(drop=True)
df_limpio.to_csv(MANIFIESTO_LIMPIO, index=False)

print(f"Imágenes evaluadas  : {n_inicial}")
print(f"Corruptas eliminadas: {eliminadas}")
print(f"Sobrevivientes      : {len(df_limpio)}")
print("\nSobrevivientes por clase:")
print(df_limpio["concepto"].value_counts())


## Celda 6 — Prevención de Data Leakage + Split Estratificado

`train_test_split(..., stratify=clase, random_state=42)` 80/20, copia física a `data/processed/{train,val}/<clase>/`. Los comentarios explican los 5 mecanismos anti-leakage.

In [ ]:
# === CELDA 6: Prevención de Data Leakage + Split Estratificado ===
import os, shutil
import pandas as pd
from sklearn.model_selection import train_test_split

SEMILLA  = 42      # fija la partición -> experimento reproducible
PROP_VAL = 0.20    # 20% validación / 80% entrenamiento

df = pd.read_csv(MANIFIESTO_LIMPIO)
# En este dataset, el "concepto" de FathomNet ES la etiqueta de clase.
df["clase"] = df["concepto"]

train_df, val_df = train_test_split(
    df,
    test_size=PROP_VAL,
    stratify=df["clase"],   # <-- clave: conserva las proporciones del desbalance
    random_state=SEMILLA,
)

# ======================================================================
# ¿CÓMO PREVIENE ESTA CELDA EL DATA LEAKAGE?
# Data leakage = que info de validación se "filtre" al entrenamiento e infle
# falsamente las métricas. Acá se evita por cinco mecanismos:
#
# 1) PARTICIÓN DISJUNTA: train_test_split manda cada imagen a UN solo lado.
#    Ninguna queda a la vez en train y val -> el modelo nunca se evalúa con
#    algo que ya vio entrenando.
# 2) UNICIDAD: en la extracción deduplicamos por UUID, así que no hay copias
#    exactas de una imagen que pudieran repartirse una en train y otra en val.
# 3) SEPARACIÓN FÍSICA: se copia a carpetas train/ y val/ separadas; la frontera
#    queda en el filesystem y es imposible mezclarlas sin querer al cargar.
# 4) SPLIT ANTES DE PROCESAR: dividimos ANTES de normalizar o aumentar. Calcular
#    estadísticas (media/desvío) sobre TODO el dataset filtraría info de val.
# 5) random_state FIJO: la partición es idéntica en cada corrida -> reproducible
#    y auditable por terceros.
#
# stratify NO balancea las clases: solo replica el desbalance real en ambos
# conjuntos para que la métrica de validación sea representativa. Atacar el
# desbalance en sí (class_weight, oversampling, etc.) es decisión de la Fase 2.
#
# LÍMITE CONOCIDO: las imágenes son framegrabs de video; cuadros consecutivos del
# mismo buceo pueden ser casi idénticos. Para rigor extremo se usaría un split por
# grupo (GroupShuffleSplit por dive/video). Queda anotado como mejora futura.
# ======================================================================

def copiar_split(df_split, particion):
    """Copia físicamente cada imagen a data/processed/<particion>/<clase>/."""
    copiadas = 0
    for _, fila in df_split.iterrows():
        destino_dir = os.path.join(DATA_PROCESSED, particion, fila["clase"])
        os.makedirs(destino_dir, exist_ok=True)
        origen  = os.path.join(DATA_RAW, fila["archivo_local"])
        destino = os.path.join(destino_dir, os.path.basename(fila["archivo_local"]))
        if os.path.exists(origen) and not os.path.exists(destino):
            shutil.copy2(origen, destino)   # copy2 conserva metadatos del archivo
            copiadas += 1
    return copiadas

copiar_split(train_df, "train")
copiar_split(val_df, "val")

print(f"Train: {len(train_df)} imágenes | Val: {len(val_df)} imágenes\n")
resumen = pd.DataFrame({
    "train": train_df["clase"].value_counts(),
    "val":   val_df["clase"].value_counts(),
}).fillna(0).astype(int)
resumen["%_val"] = (resumen["val"] / (resumen["train"] + resumen["val"]) * 100).round(1)
print(resumen)


## Celda 7 — Sanity Check Visual

Grid 4×4 de imágenes de `train/` con un CenterCrop 224×224 estático, para auditar que la fauna no quede irreconocible tras el recorte cuadrado.

In [ ]:
# === CELDA 7: Sanity Check Visual (¿qué verá la red neuronal?) ===
import os, glob, random
import matplotlib.pyplot as plt
from PIL import Image

TARGET = 224   # tamaño de entrada típico de una CNN preentrenada (ImageNet)

def center_crop_cuadrado(im, size=TARGET):
    """Simula el preprocesamiento estándar tipo ImageNet:
    escala el lado más corto a `size` y recorta el CUADRADO central.
    Sirve para auditar el riesgo del recorte cuadrado: en frames panorámicos
    (16:9 de ROV) se descartan los bordes laterales, donde la fauna podría estar."""
    im = im.convert("RGB")
    w, h = im.size
    escala = size / min(w, h)
    im = im.resize((round(w * escala), round(h * escala)))
    w, h = im.size
    izq, arr = (w - size) // 2, (h - size) // 2
    return im.crop((izq, arr, izq + size, arr + size))

# Reunir todas las imágenes de train con su etiqueta (la carpeta = la clase).
rutas = []
train_dir = os.path.join(DATA_PROCESSED, "train")
for clase in sorted(os.listdir(train_dir)):
    for r in glob.glob(os.path.join(train_dir, clase, "*.jpg")):
        rutas.append((r, clase))

muestra = random.sample(rutas, min(16, len(rutas)))

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (ruta, clase) in zip(axes.ravel(), muestra):
    ax.imshow(center_crop_cuadrado(Image.open(ruta)))
    ax.set_title(clase, fontsize=10)
    ax.axis("off")
for ax in axes.ravel()[len(muestra):]:   # apagar ejes sobrantes
    ax.axis("off")
plt.suptitle(f"Sanity check — CenterCrop {TARGET}x{TARGET} sobre data/processed/train", fontsize=14)
plt.tight_layout()
plt.show()
